## Chargement des données et répartition des splits

On charge le dataset gold (une ligne = une fenêtre temporelle par machine), puis on affiche le nombre et le pourcentage d'observations par valeur de `split_set` (train / validation / test) pour vérifier la répartition du split temporel.

In [15]:
import pandas as pd

df = pd.read_csv("artifacts/ingestions/gold/gold_splited_dataset.csv")
df.head()

split_counts = df["split_set"].value_counts()
split_pct = df["split_set"].value_counts(normalize=True) * 100

for split_name in split_counts.index:
    print(f"{split_name}: {split_counts[split_name]} ({split_pct[split_name]:.2f}%)")

train: 93990 (70.00%)
validation: 20145 (15.00%)
test: 20145 (15.00%)


## Suppression des colonnes de fuite, de l'identifiant machine, des dates et définition de la cible

On retire les colonnes de fuite `future_incident_count_*` (comptent des incidents futurs, inconnus au moment de la prédiction) ainsi que les labels `label_failure_next_*` pour tous les horizons sauf 24h. On retire également `machine_id_std` (identifiant), `window_start`/`window_end` (dates, non utilisables telles quelles comme features numériques) et `days_since_last_maintenance`. On conserve `label_failure_next_24h` comme cible et on la renomme en `y`.

In [16]:
leakage_cols = [
    "future_incident_count_6h",
    "future_incident_count_12h",
    "future_incident_count_24h",
    "future_incident_count_48h",
    "label_failure_next_6h",
    "label_failure_next_12h",
    "label_failure_next_48h",
]
id_cols = [
    "machine_id_std",
]
date_cols = [
    "window_start",
    "window_end",
]
other_cols = [
    "days_since_last_maintenance",
]

df = df.drop(columns=leakage_cols + id_cols + date_cols + other_cols)
df = df.rename(columns={"label_failure_next_24h": "y"})
df.head()

,temp_mean_1h,temp_max_1h,pressure_mean_1h,pressure_max_1h,voltage_mean_1h,voltage_max_1h,rotation_mean_1h,rotation_max_1h,pieces_produced_sum_1h,temp_mean_6h,...,type_vibration_count_prev_24h,type_bruit_mecanique_count_prev_24h,type_surconsommation_count_prev_24h,type_blocage_mecanique_count_prev_24h,type_alarme_capteur_count_prev_24h,type_arret_urgence_count_prev_24h,type_defaut_qualite_count_prev_24h,maintenance_count_prev_30d,y,split_set
0,46.340,46.340,198.203,198.203,227.568,227.568,1541.787,1541.787,4,46.3400,...,0,0,0,0,0,0,0,0,False,train
1,48.762,48.762,198.295,198.295,227.480,227.480,1537.860,1537.860,4,47.5510,...,0,0,0,0,0,0,0,0,False,train
2,51.352,51.352,199.545,199.545,228.680,228.680,1584.660,1584.660,13,48.8180,...,0,0,0,0,0,0,0,0,False,train
3,49.512,49.512,201.641,201.641,228.440,228.440,1588.960,1588.960,10,48.9915,...,0,0,0,0,0,0,0,0,False,train
4,51.982,51.982,200.157,200.157,227.840,227.840,1548.660,1548.660,6,49.5896,...,0,0,0,0,0,0,0,0,False,train


## Split temporel train / validation / test

On utilise le split temporel déjà fourni par la colonne `split_set` (train < validation < test) pour créer trois DataFrames distincts, sans mélange aléatoire.

In [17]:
df_train = df[df["split_set"] == "train"].drop(columns=["split_set"])
df_validation = df[df["split_set"] == "validation"].drop(columns=["split_set"])
df_test = df[df["split_set"] == "test"].drop(columns=["split_set"])
df = df.drop(columns=["split_set"])

print(f"train: {df_train.shape}")
print(f"validation: {df_validation.shape}")
print(f"test: {df_test.shape}")

train: (93990, 88)
validation: (20145, 88)
test: (20145, 88)


## Vérification du déséquilibre de la cible

On mesure le déséquilibre de `y` uniquement sur `df_train`, pour ne pas se baser sur validation/test lors des choix de modélisation.

In [18]:
y_counts = df_train["y"].value_counts()
y_pct = df_train["y"].value_counts(normalize=True) * 100

for label in y_counts.index:
    print(f"{label}: {y_counts[label]} ({y_pct[label]:.2f}%)")

False: 78392 (83.40%)
True: 15598 (16.60%)


## Vérification des valeurs manquantes avant standardisation

On vérifie, pour chaque colonne de `df_train` (avant standardisation), le nombre de NaN et la moyenne réelle de la colonne.

In [19]:
nan_counts = df_train.isna().sum()

for col in nan_counts.index:
    if nan_counts[col] > 0:
        print(f"{col}: {nan_counts[col]} NaN (mean: {df_train[col].mean():.2f})")
    else:
        print(f"{col}: {nan_counts[col]} NaN")

temp_mean_1h: 0 NaN
temp_max_1h: 0 NaN
pressure_mean_1h: 0 NaN
pressure_max_1h: 0 NaN
voltage_mean_1h: 0 NaN
voltage_max_1h: 0 NaN
rotation_mean_1h: 662 NaN (mean: 1589.32)
rotation_max_1h: 662 NaN (mean: 1589.32)
pieces_produced_sum_1h: 0 NaN
temp_mean_6h: 0 NaN
temp_max_6h: 0 NaN
temp_std_6h: 15 NaN (mean: 2.12)
pressure_mean_6h: 0 NaN
pressure_max_6h: 0 NaN
pressure_std_6h: 15 NaN (mean: 1.16)
voltage_mean_6h: 0 NaN
voltage_max_6h: 0 NaN
voltage_std_6h: 15 NaN (mean: 0.73)
rotation_mean_6h: 249 NaN (mean: 1589.29)
rotation_max_6h: 249 NaN (mean: 1620.71)
rotation_std_6h: 410 NaN (mean: 23.95)
temp_mean_12h: 0 NaN
temp_max_12h: 0 NaN
temp_std_12h: 15 NaN (mean: 3.35)
pressure_mean_12h: 0 NaN
pressure_max_12h: 0 NaN
pressure_std_12h: 15 NaN (mean: 1.50)
voltage_mean_12h: 0 NaN
voltage_max_12h: 0 NaN
voltage_std_12h: 15 NaN (mean: 0.96)
rotation_mean_12h: 15 NaN (mean: 1589.23)
rotation_max_12h: 15 NaN (mean: 1636.11)
rotation_std_12h: 68 NaN (mean: 29.58)
temp_mean_24h: 0 NaN
temp_max

## Imputation des valeurs manquantes par la moyenne

On remplace les NaN de `df_train` par la moyenne de chaque colonne, calculée sur train uniquement.

In [20]:
df_train = df_train.fillna(df_train.mean(numeric_only=True))

## Imputation des valeurs manquantes de validation et test

On impute les NaN de `df_validation` et `df_test` avec les moyennes calculées sur `df_train` uniquement (pas de fuite).

In [21]:
train_means = df_train.mean(numeric_only=True)

df_validation = df_validation.fillna(train_means)
df_test = df_test.fillna(train_means)

In [22]:
non_feature_cols = ["y"]
feature_cols = [col for col in df_train.columns if col not in non_feature_cols]

## Vérification des valeurs manquantes après imputation

On vérifie, pour chaque colonne de `df_train`, qu'il ne reste plus de NaN avant d'entraîner le modèle.

In [23]:
nan_counts = df_train.isna().sum()

for col in nan_counts.index:
    if nan_counts[col] > 0:
        print(f"{col}: {nan_counts[col]} NaN (mean: {df_train[col].mean():.2f})")
    else:
        print(f"{col}: {nan_counts[col]} NaN")

temp_mean_1h: 0 NaN
temp_max_1h: 0 NaN
pressure_mean_1h: 0 NaN
pressure_max_1h: 0 NaN
voltage_mean_1h: 0 NaN
voltage_max_1h: 0 NaN
rotation_mean_1h: 0 NaN
rotation_max_1h: 0 NaN
pieces_produced_sum_1h: 0 NaN
temp_mean_6h: 0 NaN
temp_max_6h: 0 NaN
temp_std_6h: 0 NaN
pressure_mean_6h: 0 NaN
pressure_max_6h: 0 NaN
pressure_std_6h: 0 NaN
voltage_mean_6h: 0 NaN
voltage_max_6h: 0 NaN
voltage_std_6h: 0 NaN
rotation_mean_6h: 0 NaN
rotation_max_6h: 0 NaN
rotation_std_6h: 0 NaN
temp_mean_12h: 0 NaN
temp_max_12h: 0 NaN
temp_std_12h: 0 NaN
pressure_mean_12h: 0 NaN
pressure_max_12h: 0 NaN
pressure_std_12h: 0 NaN
voltage_mean_12h: 0 NaN
voltage_max_12h: 0 NaN
voltage_std_12h: 0 NaN
rotation_mean_12h: 0 NaN
rotation_max_12h: 0 NaN
rotation_std_12h: 0 NaN
temp_mean_24h: 0 NaN
temp_max_24h: 0 NaN
temp_std_24h: 0 NaN
pressure_mean_24h: 0 NaN
pressure_max_24h: 0 NaN
pressure_std_24h: 0 NaN
voltage_mean_24h: 0 NaN
voltage_max_24h: 0 NaN
voltage_std_24h: 0 NaN
rotation_mean_24h: 0 NaN
rotation_max_24h: 0 N

## Entraînement de la régression logistique

On entraîne une régression logistique sur `df_train`, avec `class_weight="balanced"` pour compenser le déséquilibre de la cible observé plus haut (16.6% de positifs).

In [24]:
from sklearn.linear_model import LogisticRegression

X_train = df_train[feature_cols]
y_train = df_train["y"]

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000)
log_reg.fit(X_train, y_train)

/Users/avallet/formation-ia-project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## Entraînement du Random Forest

On entraîne un `RandomForestClassifier` sur `df_train`, avec `class_weight="balanced"` pour la même raison que la régression logistique.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(class_weight="balanced", random_state=42)
random_forest.fit(X_train, y_train)

## Entraînement du XGBoost

On entraîne un `XGBClassifier` sur `df_train`. XGBoost ne connaît pas `class_weight` : on gère le déséquilibre avec `scale_pos_weight`, calculé comme le ratio négatifs/positifs sur train.

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == False).sum() / (y_train == True).sum()

xgboost_model = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42)
xgboost_model.fit(X_train, y_train)

## Validation temporelle (TimeSeriesSplit)

On valide la robustesse des 3 modèles avec `TimeSeriesSplit` sur `df_train` : chaque fold entraîne sur le passé et valide sur une période future, respectant l'ordre chronologique (contrairement à une k-fold classique qui mélangerait les données).

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score

tscv = TimeSeriesSplit(n_splits=5)

cv_results = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    for model_name, model in [
        ("Logistic Regression", LogisticRegression(class_weight="balanced", max_iter=1000)),
        ("Random Forest", RandomForestClassifier(class_weight="balanced", random_state=42)),
        ("XGBoost", XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42)),
    ]:
        model.fit(X_fold_train, y_fold_train)
        y_proba = model.predict_proba(X_fold_val)[:, 1]
        cv_results.append({
            "fold": fold,
            "model": model_name,
            "pr_auc": average_precision_score(y_fold_val, y_proba),
        })

pd.DataFrame(cv_results).pivot(index="fold", columns="model", values="pr_auc")

## Extraction des features et de la cible pour validation et test

In [ ]:
X_validation, y_validation = df_validation[feature_cols], df_validation["y"]
X_test, y_test = df_test[feature_cols], df_test["y"]

## PR-AUC des 3 modèles sur validation et test

In [ ]:
from sklearn.metrics import average_precision_score

pr_auc_results = []
for split_name, X, y_true in [
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
]:
    for model_name, model in [
        ("Logistic Regression", log_reg),
        ("Random Forest", random_forest),
        ("XGBoost", xgboost_model),
    ]:
        y_proba = model.predict_proba(X)[:, 1]
        pr_auc_results.append({
            "split": split_name,
            "model": model_name,
            "pr_auc": average_precision_score(y_true, y_proba),
        })

pd.DataFrame(pr_auc_results).pivot(index="model", columns="split", values="pr_auc")

## Matrice de confusion à un seuil choisi

On fixe un seuil de décision explicite (`threshold`, par défaut 0.5) et on affiche la matrice de confusion des 3 modèles sur `df_validation`, pour visualiser le compromis faux négatifs (pannes manquées) / faux positifs (fausses alertes).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

threshold = 0.5
labels = [["TN", "FP"], ["FN", "TP"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, model) in zip(axes, [
    ("Logistic Regression", log_reg),
    ("Random Forest", random_forest),
    ("XGBoost", xgboost_model),
]):
    y_pred = (model.predict_proba(X_validation)[:, 1] >= threshold).astype(int)
    cm = confusion_matrix(y_validation, y_pred)

    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]}", ha="center", va="center")

    ax.set_xticks([0, 1], ["False", "True"])
    ax.set_yticks([0, 1], ["False", "True"])
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(f"{model_name}\n(seuil={threshold})")

plt.tight_layout()
plt.show()

## Recall des 3 modèles au seuil choisi

On calcule le recall (rappel) de chaque modèle sur `df_validation`, au même seuil que la matrice de confusion — la part des vraies pannes effectivement détectées.

In [37]:
from sklearn.metrics import recall_score

recall_results = []
for model_name, model in [
    ("Logistic Regression", log_reg),
    ("Random Forest", random_forest),
    ("XGBoost", xgboost_model),
]:
    y_pred = (model.predict_proba(X_validation)[:, 1] >= threshold).astype(int)
    recall_results.append({
        "model": model_name,
        "recall": recall_score(y_validation, y_pred),
    })

pd.DataFrame(recall_results).set_index("model")

,recall
model,
Logistic Regression,0.447581
Random Forest,0.421371
XGBoost,0.423099
